# Hello Pipeline — REAL-DATA PILOT (A1 + D1, Kaggle)

**Purpose:** run the `hello.md` pipeline end-to-end on **real** inputs to see actual results. This is a **pilot**: the numbers are real, but three engine caveats from the audit remain in the code as-is, so tiers are **provisional**:

- Engine 2's "S-LDSC" is the code's internal OLS proxy (no real S-LDSC run).
- The RSR audit does not apply the $P_\perp$ projection; Engine 3 spot p-values use the code's own null handling.
- ADT CLR is per-protein across spots (the code's convention).

**Real inputs:** MAGMA z-scores computed in this notebook from the locked GWAS files (16 traits; FL and DLBCL separate — no combined GWAS exists), GENCODE v44 gene annotation from the repo, covariates from data (gene length, median SNP count, pooled mean expression), Muse-GCN seed-42 embeddings, n_perm = 1000 (the code recalibrates to 10000 itself if a negative control is Confirmed).

**Attach before running:** datasets `pulokpulok/spatial-multiomics-6datasets-private` and `pulokpulok/hello-pipeline-refs`, secret `GITHUB_TOKEN`, Internet ON. Then **Save Version -> Run All** (headless runs survive disconnects; interactive sessions do not).

**Runtime estimate:** MAGMA ~0.5-2.5 h for 16 traits; pipeline ~1-3 h for both sections at n_perm=1000. If the negative-control gate triggers the 10000-permutation recalibration, add several hours. Checkpoints are pushed to GitHub (`runs/hello-pilot-real`) after MAGMA, after A1, and after D1, so partial progress survives.

**Reading the result:** check the negative controls (Adult_Standing_Height, Heel_BMD, Hair_Colour) FIRST — if they light up, the discovery hits are background, not signal.


In [ ]:
import os, sys, subprocess
for pkg in ["scanpy", "statsmodels"]:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=True)
import numpy as np
import pandas as pd
import scanpy as sc
print("imports ok; numpy", np.__version__, "| pandas", pd.__version__)


In [ ]:
# ---------- CONFIG (real pilot, locked 2026-11-10) ----------
SEED = 42
REAL_N_PERM = 1000           # code default scale; recalibration to 10000 is the code's own gate
RAW_BASE = "https://raw.githubusercontent.com/rifatahsanpul0k/my_research/main"
WORK = "/kaggle/working/pilot_inputs"
OUTDIR = "/kaggle/working/pilot_outputs"
REFS = "/kaggle/working/refs"
MAGMA_WORK = "/kaggle/working/magma_work"
for d in (WORK, OUTDIR, REFS, MAGMA_WORK):
    os.makedirs(d, exist_ok=True)

SECTIONS = {
    "A1": {"folder": "10x_human_lymph_node_A1",
           "emb": "10x_human_lymph_node_A1_seed42_embeddings.npy",
           "lab": "10x_human_lymph_node_A1_seed42_labels.npy"},
    "D1": {"folder": "10x_human_lymph_node_D1",
           "emb": "10x_human_lymph_node_D1_seed42_embeddings.npy",
           "lab": "10x_human_lymph_node_D1_seed42_labels.npy"},
}

import glob as _glob
def find_refs_root():
    cands = []
    for pat in ("/kaggle/input/*", "/kaggle/input/*/*", "/kaggle/input/*/*/*"):
        cands += _glob.glob(pat)
    for root in cands:
        if os.path.exists(os.path.join(root, "magma.zip")) or os.path.isdir(os.path.join(root, "magma")):
            return root
    return None

REFS_IN = find_refs_root()
assert REFS_IN, "hello-pipeline-refs dataset not attached (no magma.zip or magma/ folder under /kaggle/input)"
print("refs dataset at:", REFS_IN)

def find_section_dir(folder):
    for pat in (f"/kaggle/input/**/{folder}/adata_RNA.h5ad",):
        hits = _glob.glob(pat, recursive=True)
        if hits:
            return os.path.dirname(hits[0])
    return None

SEC_DIRS = {}
for sec, spec in SECTIONS.items():
    d = find_section_dir(spec["folder"])
    assert d, f"{spec['folder']} not found - attach pulokpulok/spatial-multiomics-6datasets-private"
    SEC_DIRS[sec] = d
    print(sec, "data:", d)

# ---------- Muse-GCN seed-42 embeddings + labels from the GitHub repo ----------
import urllib.request
EMB = {}
for sec, spec in SECTIONS.items():
    paths = {}
    for kind, name in (("E", spec["emb"]), ("C", spec["lab"])):
        dst = os.path.join(WORK, name)

        if not os.path.exists(dst):
            urllib.request.urlretrieve(f"{RAW_BASE}/embeddings/{name}", dst)
        paths[kind] = np.load(dst)
    assert paths["E"].shape[0] == paths["C"].shape[0], f"{sec}: E rows != C length"
    EMB[sec] = paths
    print(sec, "E:", paths["E"].shape, "| domains:", np.unique(paths["C"]))


In [ ]:
# ---------- GitHub checkpoint push helper (runs/hello-pilot-real) ----------
# Failures only warn: outputs also stay in /kaggle/working/pilot_outputs and in
# the Kaggle version output. Token comes from the GITHUB_TOKEN Kaggle secret.
import subprocess, base64, shutil
def push_outputs(message):
    try:
        from kaggle_secrets import UserSecretsClient
        _tok = UserSecretsClient().get_secret("GITHUB_TOKEN")
        _hdr = "Authorization: Basic " + base64.b64encode(f"x-access-token:{_tok}".encode()).decode()
        _GIT = ["git", "-c", f"http.extraHeader={_hdr}"]
        _REPO = "/kaggle/working/my_research_push"
        if os.path.isdir(_REPO):
            shutil.rmtree(_REPO)
        subprocess.run(_GIT + ["clone", "--depth", "1", "--filter=blob:none", "--sparse",
                               "https://github.com/rifatahsanpul0k/my_research.git", _REPO],
                       check=True, capture_output=True, text=True)
        subprocess.run(["git", "sparse-checkout", "set", "runs/hello-pilot-real"],
                       cwd=_REPO, check=True, capture_output=True, text=True)
        _dst = os.path.join(_REPO, "runs", "hello-pilot-real")
        os.makedirs(_dst, exist_ok=True)
        for fn in os.listdir(OUTDIR):
            src = os.path.join(OUTDIR, fn)
            if os.path.isfile(src):
                shutil.copy(src, os.path.join(_dst, fn))
            elif os.path.isdir(src):
                shutil.copytree(src, os.path.join(_dst, fn), dirs_exist_ok=True)
        def _sh(*a):
            return subprocess.run(list(a), cwd=_REPO, capture_output=True, text=True)
        _sh("git", "config", "user.name", "kaggle-runner")
        _sh("git", "config", "user.email", "kaggle-runner@local")
        _sh("git", "add", "runs/hello-pilot-real")
        if _sh("git", "diff", "--cached", "--quiet").returncode != 0:
            _sh("git", "commit", "-m", message)
            subprocess.run(_GIT + ["pull", "--rebase", "origin", "main"], cwd=_REPO, capture_output=True, text=True)
            _r = subprocess.run(_GIT + ["push", "origin", "main"], cwd=_REPO, capture_output=True, text=True)
            print("pushed:", message if _r.returncode == 0 else "FAILED " + _r.stderr[:300])
        else:
            print("push: nothing new")
    except Exception as e:
        print("push skipped/failed (outputs safe in pilot_outputs):", str(e)[:300])


In [ ]:
# ---------- Reference bundles + MAGMA binary (zip OR Kaggle-extracted layout) ----------
import zipfile, shutil, urllib.request, stat
def _find1(pat):
    hits = _glob.glob(pat, recursive=True)
    assert hits, f"not found: {pat}"
    return hits[0]

def _maybe(pat):
    hits = _glob.glob(pat, recursive=True)
    return hits[0] if hits else None

def _unzip(zpath, dest):
    with zipfile.ZipFile(zpath) as zf:
        zf.extractall(dest)

# Layout A: Kaggle already extracted the bundles (folders, no .zip files).
# Layout B: bundles still .zip -> extract into REFS (working dir).
if _maybe(os.path.join(REFS_IN, "**", "NCBI38.gene.loc")):
    print("refs layout: already extracted by Kaggle - using in place")
    SEARCH_ROOTS = [REFS_IN]
else:
    print("refs layout: zip bundles - extracting")
    for z in ("magma.zip", "gwas.zip"):
        zp = os.path.join(REFS_IN, z)
        assert os.path.exists(zp), f"{z} not found in refs dataset root {REFS_IN}"
        _unzip(zp, REFS)
    for inner, dest in (("magma_v1.10_static.zip", "magma_bin"),
                        ("NCBI38.zip", "gene_loc"),
                        ("g1000_eur.zip", "g1000")):
        _unzip(_find1(os.path.join(REFS, "**", inner)), os.path.join(REFS, dest))
    SEARCH_ROOTS = [REFS]

def _find_any(pat):
    for r in SEARCH_ROOTS:
        h = _maybe(os.path.join(r, "**", pat))
        if h:
            return h
    raise AssertionError(f"not found under {SEARCH_ROOTS}: {pat}")

# MAGMA binary: copy to the working dir so it can be made executable
# (/kaggle/input is read-only).
_bin_src = _find_any(os.path.join("magma_v1.10_static", "magma"))
MAGMA = os.path.join(REFS, "magma_bin", "magma")
os.makedirs(os.path.dirname(MAGMA), exist_ok=True)
if os.path.abspath(_bin_src) != os.path.abspath(MAGMA):
    shutil.copy(_bin_src, MAGMA)
os.chmod(MAGMA, os.stat(MAGMA).st_mode | stat.S_IEXEC)

GENE_LOC_38 = _find_any("NCBI38.gene.loc")
BFILE = _find_any("g1000_eur.bim")[:-4]  # strip .bim
assert os.path.exists(BFILE + ".bed") and os.path.exists(BFILE + ".fam"), "g1000_eur plink files incomplete"

# NCBI37.3 gene locations were not in the download bundle - fetch (0.35 MB, link verified 2026-11-10)
_ncbi37 = os.path.join(REFS, "NCBI37.3.zip")
if not os.path.exists(_ncbi37):
    urllib.request.urlretrieve("https://vu.data.surfsara.nl/index.php/s/Pj2orwuF2JYyKxq/download", _ncbi37)
_unzip(_ncbi37, os.path.join(REFS, "gene_loc"))
GENE_LOC_37 = _find1(os.path.join(REFS, "gene_loc", "**", "NCBI37.3.gene.loc"))

_r = subprocess.run([MAGMA], capture_output=True, text=True)
print("MAGMA binary responds:", (("MAGMA" in _r.stdout) or ("magma" in _r.stdout.lower())))
print("BFILE:", BFILE)
print("GENE_LOC_38:", GENE_LOC_38)
print("GENE_LOC_37:", GENE_LOC_37)
# locate the gwas folder (holds one subfolder per trait)
_cand = []
for r in SEARCH_ROOTS:
    _cand += [d for d in _glob.glob(os.path.join(r, "**", "SLE"), recursive=True) if os.path.isdir(d)]
assert _cand, "gwas/SLE folder not found"
GWAS_ROOT = os.path.dirname(_cand[0])
print("GWAS_ROOT:", GWAS_ROOT)


In [ ]:
# ---------- Sumstats -> MAGMA inputs (snp-loc + pval with N column) ----------
# Column mappings verified against the live file headers on 2026-11-10.
# build: '38' -> NCBI38 gene locations, '37' -> NCBI37.3 (annotation uses each
# file's OWN chr/pos; gene analysis matches the EUR reference by rsID).
# MS and Hair_Colour carry no rsID column -> mapped through the g1000_eur .bim
# (chr:pos, GRCh37 - both files are GRCh37, so the mapping is build-consistent).
TRAIT_SPEC = {
 # key                    folder               file                                                              sep     rsid          chr             pos                    p / neglog10        N fixed  N col  build
 "SLE":                   ("SLE",               "bentham_2015_26502338_sle_efo0002690_1_gwas.sumstats.tsv.gz",       "\t",  "rsid",       "chrom",        "pos",                 "p",                23210,   None,  "37"),
 "RA":                    ("RA",                "GCST90132223.h.tsv.gz",                                               "\t",  "rsid",       "chromosome",   "base_pair_location",  "p_value",          97173,   None,  "38"),
 "IBD":                   ("IBD",               "28067908-GCST004131-EFO_0003767.h.tsv.gz",                              "\t",  "variant_id", "chromosome",   "base_pair_location",  "p_value",          59957,   None,  "38"),
 "MS":                    ("MS",                "GCST90558092.tsv",                                                    "\t",  None,         "chromosome",   "base_pair_location",  "p_value",          360004,  None,  "37"),
 "T1D":                   ("T1D",               "34012112-GCST90014023-EFO_0001359.h.tsv.gz",                            "\t",  "variant_id", "chromosome",   "base_pair_location",  "p_value",          520580,  None,  "38"),
 "Celiac":                ("CELIAC",            "20190752-GCST000612-EFO_0001060.h.tsv.gz",                              "\t",  "variant_id", "chromosome",   "base_pair_location",  "p_value",          15283,   None,  "38"),
 "DLBCL":                 ("DLBCL",             "GCST90624739.h.tsv.gz",                                               "\t",  "rsid",       "chromosome",   "base_pair_location",  "p_value",          658535,  None,  "38"),
 "FL":                    ("FL",                "GCST90624741.h.tsv.gz",                                               "\t",  "rsid",       "chromosome",   "base_pair_location",  "p_value",          657982,  None,  "38"),
 "CLL":                   ("CLL",               "GCST90624738.h.tsv.gz",                                               "\t",  "rsid",       "chromosome",   "base_pair_location",  "p_value",          1108777, None,  "38"),
 "Lymphocyte_Count":      ("LYMPHOCYTE_COUNT",  "32888493-GCST90002316-EFO_0004587.h.tsv.gz",                            "\t",  "variant_id", "chromosome",   "base_pair_location",  "p_value",          524923,  None,  "38"),
 "Adult_Standing_Height": ("HEIGHT",            "GIANT_HEIGHT_YENGO_2022_GWAS_SUMMARY_STATS_EUR.gz",                   "\t",  "RSID",       "CHR",          "POS",                 "P",                None,    "N",   "37"),
 "Heel_BMD":              ("HEEL_BMD",          "30598549-GCST006979-EFO_0009270.h.tsv.gz",                              "\t",  "variant_id", "chromosome",   "base_pair_location",  "p_value",          None,    "n",   "38"),
 "Hair_Colour":           ("HAIR_COLOUR",       "hair_panukbb_1747_blonde.tsv.bgz",                                    "\t",  None,         "chr",          "pos",                 "neglog10_pval_EUR",419469,  None,  "37"),
 "ASD":                   ("ASD",               "iPSYCH-PGC_ASD_Nov2017.gz",                                           "\t",  "SNP",        "CHR",          "BP",                  "P",                46350,   None,  "37"),
 "Cortical_Surface_Area": ("CORTICAL_SURFACE_AREA", "ENIGMA3_mixed_se_wo_Mean_Full_SurfArea_20190429.txt.gz",          "\\s+", "SNP",        "CHR",          "BP",                  "P",                None,    "N",   "37"),
 "Cortical_Thickness":    ("CORTICAL_THICKNESS",    "ENIGMA3_mixed_se_wo_Mean_Full_Thickness_20190429.txt.gz",          "\\s+", "SNP",        "CHR",          "BP",                  "P",                None,    "N",   "37"),
}

BIM = None
def load_bim():
    global BIM
    if BIM is None:
        bim = pd.read_csv(BFILE + ".bim", sep="\t", header=None,
                          names=["CHR", "SNP", "CM", "BP", "A1", "A2"], dtype={"CHR": str, "SNP": str, "BP": np.int64})
        BIM = bim.drop_duplicates(subset=["CHR", "BP"])[["CHR", "BP", "SNP"]]
    return BIM

def prep_trait(key, spec):
    folder, fname, sep, rsid_c, chr_c, pos_c, p_c, n_fixed, n_col, build = spec
    outdir = os.path.join(MAGMA_WORK, key)
    os.makedirs(outdir, exist_ok=True)
    snp_loc_f = os.path.join(outdir, key + ".snp_loc.txt")
    pval_f = os.path.join(outdir, key + ".pval.txt")
    if os.path.exists(snp_loc_f) and os.path.exists(pval_f):
        print(key, ": already prepped"); return snp_loc_f, pval_f, build
    path = os.path.join(GWAS_ROOT, folder, fname)
    if not os.path.exists(path):
        # fall back: find the file anywhere under GWAS_ROOT (folder/name drift)
        _hits = [h for h in _glob.glob(os.path.join(GWAS_ROOT, "**", fname), recursive=True)
                 if os.path.isfile(h)]
        if not _hits:
            _acc = next((t for t in fname.replace("_", "-").split("-")
                         if t.startswith("GCST")), None)
            if _acc:
                _hits = [h for h in _glob.glob(os.path.join(GWAS_ROOT, "**", "*" + _acc + "*"),
                                               recursive=True) if os.path.isfile(h)]
        if _hits:
            path = _hits[0]
            print(f"{key}: resolved by search -> {path}")
    assert os.path.exists(path), f"{key}: sumstats file missing: {path}"
    print(f"{key}: reading {path}")
    want = {chr_c, pos_c, p_c} | ({rsid_c} if rsid_c else set()) | ({n_col} if n_col else set())
    with open(path, "rb") as _fh:
        _comp = "gzip" if _fh.read(2) == bytes([0x1F, 0x8B]) else None  # trust magic bytes, not the name
    df = pd.read_csv(path, sep=sep, engine="python" if sep == "\\s+" else "c", compression=_comp,
                     usecols=lambda c: c in want or c == "low_confidence_EUR")
    df = df.rename(columns={chr_c: "CHR", pos_c: "BP", p_c: "PRAW"})
    if key == "Hair_Colour":
        if "low_confidence_EUR" in df.columns:
            df = df[df["low_confidence_EUR"].astype(str).str.lower() == "false"]
        df["P"] = np.power(10.0, -df["PRAW"].astype(float))
    else:
        df["P"] = df["PRAW"].astype(float)
    df["CHR"] = df["CHR"].astype(str).str.replace("^chr", "", regex=True)
    df["BP"] = pd.to_numeric(df["BP"], errors="coerce")
    df = df.dropna(subset=["BP", "P"])
    df["BP"] = df["BP"].astype(np.int64)
    if rsid_c:
        df["SNP"] = df[rsid_c].astype(str)
    else:
        bim = load_bim()
        df = df.merge(bim, on=["CHR", "BP"], how="inner")
        print(f"{key}: bim-mapped {len(df)} rows")
    df = df[df["SNP"].str.startswith("rs", na=False)]
    df["P"] = df["P"].clip(lower=1e-300, upper=1.0)
    df["N"] = pd.to_numeric(df[n_col], errors="coerce").fillna(n_fixed) if n_col else float(n_fixed)
    df = df.sort_values("P").drop_duplicates(subset=["SNP"], keep="first")
    df[["SNP", "CHR", "BP"]].to_csv(snp_loc_f, sep="\t", index=False, header=False)
    df[["SNP", "P", "N"]].to_csv(pval_f, sep="\t", index=False)
    print(f"{key}: {len(df)} SNPs prepped (build {build})")
    return snp_loc_f, pval_f, build

PREPPED = {}
for _k, _spec in TRAIT_SPEC.items():
    PREPPED[_k] = prep_trait(_k, _spec)
print("prepped traits:", len(PREPPED))


In [ ]:
# ---------- MAGMA: annotate + gene analysis per trait (checkpointed) ----------
import time
MAGMA_OUT = os.path.join(OUTDIR, "magma")
os.makedirs(MAGMA_OUT, exist_ok=True)
GENES_OUT = {}
for key, (snp_loc_f, pval_f, build) in PREPPED.items():
    gene_loc = GENE_LOC_38 if build == "38" else GENE_LOC_37
    prefix = os.path.join(MAGMA_WORK, key, key)
    t0 = time.time()
    if not os.path.exists(prefix + ".genes.annot"):
        r = subprocess.run([MAGMA, "--annotate", "window=10,1.5", "--snp-loc", snp_loc_f,
                            "--gene-loc", gene_loc, "--out", prefix],
                           capture_output=True, text=True)
        assert r.returncode == 0, f"{key} annotate failed: {r.stdout[-500:]} {r.stderr[-500:]}"
    if not os.path.exists(prefix + ".genes.out"):
        r = subprocess.run([MAGMA, "--bfile", BFILE, "--pval", pval_f, "ncol=N",
                            "--gene-annot", prefix + ".genes.annot", "--out", prefix],
                           capture_output=True, text=True)
        assert r.returncode == 0, f"{key} gene analysis failed: {r.stdout[-800:]} {r.stderr[-500:]}"
    dst = os.path.join(MAGMA_OUT, key + ".genes.out")
    shutil.copy(prefix + ".genes.out", dst)
    GENES_OUT[key] = dst
    if os.path.exists(prefix + ".genes.annot"):
        os.remove(prefix + ".genes.annot")  # reclaim disk; re-annotate on rerun if needed
    print(f"{key}: genes.out ok ({(time.time()-t0)/60:.1f} min)")
print("MAGMA done:", len(GENES_OUT), "traits")
push_outputs("pilot-real: MAGMA genes.out for 16 traits (A1+D1 run)")


In [ ]:
# ---------- PREP A1 + D1: REAL gene annotation join + ADT/annotation alignment ----------
# Annotation: repo GENCODE v44 table (gene_id, gene_symbol, chrom, start, end, biotype).
# Join on the h5ad var['gene_ids'] (Ensembl), unique-symbol fallback, as verified
# in the Step-2 prep (18,079 / 18,085 matched). Unmatched genes get biotype
# 'unannotated' + chrom '.' so Stage 1 drops them.
import urllib.request
_anno_csv = os.path.join(WORK, "gene_annotation_GRCh38_gencode_v44.csv")
if not os.path.exists(_anno_csv):
    urllib.request.urlretrieve(f"{RAW_BASE}/resources/gene_annotation_GRCh38_gencode_v44.csv", _anno_csv)
ANNO = pd.read_csv(_anno_csv)
_by_id = {c: dict(zip(ANNO["gene_id"].astype(str), ANNO[c])) for c in ("chrom", "start", "end", "biotype")}
_by_sym = {c: dict(zip(ANNO["gene_symbol"].astype(str), ANNO[c])) for c in ("chrom", "start", "end", "biotype")}

SEC_PREP = {}
ADATAS = {}
for sec, spec in SECTIONS.items():
    sdir = SEC_DIRS[sec]
    rna_path = os.path.join(sdir, "adata_RNA.h5ad")
    adt_path = os.path.join(sdir, "adata_ADT.h5ad")
    anno_path = os.path.join(sdir, "annotation.csv")
    E, C = EMB[sec]["E"], EMB[sec]["C"]
    adata_rna = sc.read_h5ad(rna_path)
    assert adata_rna.n_obs == E.shape[0], f"{sec}: RNA spots != embedding rows - ALIGNMENT BROKEN, STOP"
    gids = adata_rna.var["gene_ids"].astype(str).values
    syms = adata_rna.var_names.astype(str).values
    for col in ("chrom", "start", "end", "biotype"):
        vals = np.array([_by_id[col].get(g, _by_sym[col].get(s)) for g, s in zip(gids, syms)], dtype=object)
        adata_rna.var[col] = vals
    n_unmatched = int(adata_rna.var["chrom"].isna().sum())
    adata_rna.var["chrom"] = adata_rna.var["chrom"].fillna(".").astype(str)
    adata_rna.var["start"] = pd.to_numeric(adata_rna.var["start"], errors="coerce").fillna(0).astype(np.int64)
    adata_rna.var["end"] = pd.to_numeric(adata_rna.var["end"], errors="coerce").fillna(0).astype(np.int64)
    adata_rna.var["biotype"] = adata_rna.var["biotype"].fillna("unannotated").astype(str)
    print(f"{sec}: annotation joined; unmatched genes: {n_unmatched} / {adata_rna.n_vars}")
    out_rna = os.path.join(WORK, f"adata_RNA_{sec}_annot.h5ad")
    adata_rna.write_h5ad(out_rna)
    ADATAS[sec] = adata_rna

    adata_adt = sc.read_h5ad(adt_path)
    assert adata_adt.n_obs == adata_rna.n_obs, f"{sec}: ADT spots != RNA spots - STOP"
    if list(adata_adt.obs_names) != list(adata_rna.obs_names):
        assert set(adata_adt.obs_names) == set(adata_rna.obs_names), f"{sec}: ADT barcodes mismatch - STOP"
        adata_adt = adata_adt[list(adata_rna.obs_names)]
        print(f"{sec}: ADT reordered to RNA spot order")
    out_adt = os.path.join(WORK, f"adata_ADT_{sec}_prep.h5ad")
    adata_adt.write_h5ad(out_adt)

    anno = pd.read_csv(anno_path, index_col=0)
    assert "manual-anno" in anno.columns, f"{sec}: annotation.csv has no manual-anno column"
    if set(anno.index.astype(str)) == set(adata_rna.obs_names.astype(str)):
        anno = anno.reindex(adata_rna.obs_names)
        assert anno["manual-anno"].notna().all(), f"{sec}: annotation reindex NaNs - STOP"
        out_anno = os.path.join(WORK, f"annotation_{sec}_aligned.csv")
        anno.to_csv(out_anno)
        print(f"{sec}: annotation aligned to RNA spot order")
    else:
        out_anno = anno_path
        print(f"{sec}: WARNING annotation index != RNA barcodes; driver will read it positionally")
    SEC_PREP[sec] = {"rna": out_rna, "adt": out_adt, "anno": out_anno}
print("section prep done:", SEC_PREP)


In [ ]:
# ---------- Build magma_z_dict + gene_covariates from MAGMA outputs ----------
import pickle
def _read_gene_loc(path):
    m = {}
    with open(path) as f:
        for line in f:
            parts = line.rstrip("\n").split("\t")
            if len(parts) >= 6:
                m[parts[0]] = parts[5]
    return m
LOC_MAP = {"38": _read_gene_loc(GENE_LOC_38), "37": _read_gene_loc(GENE_LOC_37)}

magma_z_dict = {}
_nsnps = {}
for key, (_, _, build) in PREPPED.items():
    go = pd.read_csv(GENES_OUT[key], sep="\\s+", engine="python")
    go["symbol"] = go["GENE"].astype(str).map(LOC_MAP[build])
    go = go.dropna(subset=["symbol"])
    go["_absz"] = go["ZSTAT"].abs()
    go = go.sort_values("_absz", ascending=False).drop_duplicates(subset=["symbol"], keep="first")
    magma_z_dict[key] = pd.Series(go["ZSTAT"].values, index=go["symbol"].values)
    _nsnps[key] = pd.Series(go["NSNPS"].values, index=go["symbol"].values)
    print(f"{key}: {len(magma_z_dict[key])} genes with z-scores")

_panel_genes = set()
for sec in SECTIONS:
    _panel_genes |= set(ADATAS[sec].var_names.astype(str))
_cov_syms = sorted(_panel_genes | set().union(*[set(s.index) for s in magma_z_dict.values()]))
_len_map = {}
for g, s, e in zip(ANNO["gene_symbol"].astype(str), ANNO["start"], ANNO["end"]):
    _len_map.setdefault(g, float(e) - float(s) + 1.0)
_nsnps_med = pd.DataFrame(_nsnps).median(axis=1, skipna=True)
_expr = {}
for sec in SECTIONS:
    X = ADATAS[sec].X
    if hasattr(X, "toarray"):
        import scipy.sparse as _sp
        Xl = X.copy()
        Xl.data = np.log1p(Xl.data)
        m = np.asarray(Xl.mean(axis=0)).ravel()
    else:
        m = np.log1p(np.asarray(X)).mean(axis=0)
    _expr[sec] = pd.Series(m, index=ADATAS[sec].var_names.astype(str))
_expr_mean = pd.DataFrame(_expr).mean(axis=1)
gene_covariates = pd.DataFrame(index=_cov_syms)
gene_covariates["log10_gene_length"] = np.log10(pd.Series(_len_map).reindex(_cov_syms).fillna(30000.0))
gene_covariates["log10_median_nsnps"] = np.log10(_nsnps_med.reindex(_cov_syms).fillna(_nsnps_med.median()))
gene_covariates["mean_log_expr"] = _expr_mean.reindex(_cov_syms).fillna(0.0)
gene_covariates = gene_covariates[~gene_covariates.index.duplicated(keep="first")]
print("covariates:", gene_covariates.shape)

with open(os.path.join(OUTDIR, "magma_z_dict.pkl"), "wb") as f:
    pickle.dump(magma_z_dict, f)
gene_covariates.to_csv(os.path.join(OUTDIR, "gene_covariates.csv"))
for key, s in magma_z_dict.items():
    cov = len(set(s.index) & _panel_genes)
    print(f"coverage {key}: {cov} / {len(_panel_genes)} panel genes")


In [ ]:
# ---------- MAIN PIPELINE CODE (verbatim from hello.md, block 1 of 2) ----------
import numpy as np
import pandas as pd
import scipy.stats as stats
from scipy.spatial.distance import cdist
from scipy.optimize import linear_sum_assignment
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score
from statsmodels.stats.multitest import multipletests
from typing import Dict, List, Tuple, Optional

# =====================================================================
# CONFIGURATION: PRE-SPECIFIED TRAIT PANELS & ANTIBODY MAPPING (STAGE 1)
# =====================================================================

TRAIT_PANELS_LYMPH_NODE = {
    "D_disc": [
        "SLE", "RA", "IBD", "MS", "T1D", "Celiac",
        "Follicular_Lymphoma_DLBCL", "CLL", "Lymphocyte_Count"
    ],
    "D_neg": [
        "Adult_Standing_Height", "Heel_BMD", "Hair_Colour"
    ],
    "D_spec": [
        "ASD", "Cortical_Surface_Area", "Cortical_Thickness"
    ]
}

# Curated Antibody-to-Gene mapping table G(p) (Stage 2.1 Branch A.2)
DEFAULT_ADT_TO_GENES = {
    "CD3": ["CD3D", "CD3E", "CD3G"],
    "CD4": ["CD4"],
    "CD8a": ["CD8A"],
    "CD19": ["CD19"],
    "CD20": ["MS4A1"],
    "CD45RA": ["PTPRC"],
    "CD45RO": ["PTPRC"],
    "PD-1": ["PDCD1"],
    "CD11c": ["ITGAX"],
    "CD14": ["CD14"],
    "CD16": ["FCGR3A", "FCGR3B"],
    "CD25": ["IL2RA"],
    "CD27": ["CD27"],
    "CD28": ["CD28"],
    "CD38": ["CD38"],
    "CD40": ["CD40"],
    "CD56": ["NCAM1"],
    "CD69": ["CD69"],
    "CD80": ["CD80"],
    "CD86": ["CD86"],
    "CD127": ["IL7R"],
    "CXCR5": ["CXCR5"],
    "HLA-DR": ["HLA-DRA", "HLA-DRB1"],
    "IgD": ["IGHD"],
    "IgM": ["IGHM"]
}


# =====================================================================
# CORE MATHEMATICAL UTILITIES (ACAT, FDR, BROWN'S METHOD, M_EFF)
# =====================================================================

def acat_combine(p_values: np.ndarray, weights: Optional[np.ndarray] = None) -> float:
    """
    Equation 4.1 & 4.2: Cauchy Combination Test (ACAT) with [1e-15, 1 - 1e-15] truncation.
    """
    p = np.clip(np.asarray(p_values, dtype=np.float64), 1e-15, 1.0 - 1e-15)
    if weights is None:
        weights = np.full(len(p), 1.0 / len(p), dtype=np.float64)
    else:
        weights = np.asarray(weights, dtype=np.float64)
        weights = weights / np.sum(weights)
    t_acat = np.sum(weights * np.tan((0.5 - p) * np.pi))
    p_comb = 0.5 - np.arctan(t_acat) / np.pi
    return float(np.clip(p_comb, 1e-15, 1.0 - 1e-15))


def acat_statistic_only(p_values: np.ndarray, weights: Optional[np.ndarray] = None) -> float:
    """Returns raw T_ACAT statistic for empirical permutation comparison."""
    p = np.clip(np.asarray(p_values, dtype=np.float64), 1e-15, 1.0 - 1e-15)
    if weights is None:
        weights = np.full(len(p), 1.0 / len(p), dtype=np.float64)
    else:
        weights = np.asarray(weights, dtype=np.float64)
        weights = weights / np.sum(weights)
    return float(np.sum(weights * np.tan((0.5 - p) * np.pi)))


def kost_empirical_browns_method(p_values: np.ndarray, corr_matrix: np.ndarray) -> float:
    """
    Stage 3.4 Branch 3B.3: Kost's Empirical Brown's Method for combining correlated p-values.
    """
    p = np.clip(np.asarray(p_values, dtype=np.float64), 1e-15, 1.0 - 1e-15)
    M0 = len(p)
    psi = -2.0 * np.sum(np.log(p))
    e_psi = 2.0 * M0
    cov_sum = 0.0
    for a in range(M0):
        for b in range(a + 1, M0):
            rho = corr_matrix[a, b]
            cov_sum += 3.263 * rho + 0.710 * (rho ** 2) + 0.027 * (rho ** 3)
    var_psi = 4.0 * M0 + 2.0 * cov_sum
    var_psi = max(var_psi, 1e-8)
    c = var_psi / (2.0 * e_psi)
    f = (2.0 * (e_psi ** 2)) / var_psi
    return float(1.0 - stats.chi2.cdf(psi / c, df=f))


def cognate_receptor_joint_z(z_rna: np.ndarray, z_adt: np.ndarray) -> Tuple[np.ndarray, float]:
    """
    Stage 3.4 Branch 3B.3: Correlation-Calibrated Cognate Receptor Diagnostics.
    """
    rho_hat, _ = stats.spearmanr(z_rna, z_adt)
    rho_hat = float(np.clip(rho_hat, -0.99, 0.99))
    z_joint = (z_rna + z_adt) / np.sqrt(2.0 * (1.0 + rho_hat))
    return z_joint, rho_hat


def compute_meff_cheverud_nyholt(p_grid_3_engines: np.ndarray) -> float:
    """
    Stage 4B.1.3: Cheverud-Nyholt Effective Number of Engines M_eff^(s).
    p_grid_3_engines has shape (N_pairs, 3).
    """
    p_clipped = np.clip(p_grid_3_engines, 1e-15, 1.0 - 1e-15)
    z_grid = stats.norm.ppf(1.0 - p_clipped)
    corr_mat = np.corrcoef(z_grid, rowvar=False)
    corr_mat = np.nan_to_num(corr_mat, nan=0.0)
    np.fill_diagonal(corr_mat, 1.0)
    eigenvalues = np.linalg.eigvalsh(corr_mat)
    # Using population variance across the 3 eigenvalues per specification
    var_lambda = np.var(eigenvalues, ddof=0)
    m_eff = 1.0 + 2.0 * (1.0 - var_lambda / 3.0)
    return float(m_eff)


# =====================================================================
# STAGE 1: WITHIN-SECTION FEATURE HARMONIZATION (BRANCH A: A1 & D1)
# =====================================================================

def stage1_harmonize_lymph_node(
    rna_counts: np.ndarray,
    gene_meta: pd.DataFrame,
    adt_counts: np.ndarray,
    adt_names: List[str]
) -> Dict[str, object]:
    """
    Stage 1 (Branch A):
    - Filters MT-*, RPL*, RPS*, non-coding, non-autosomal, and <1% spot detection.
    - Generates Primary (+MHC) and Sensitivity (-MHC: masks chr6:25,000,000-34,000,000) tracks.
    - Log-normalizes RNA counts and CLR-normalizes ADT counts across spots.
    
    gene_meta required columns: ['hgnc_symbol', 'chrom', 'start', 'end', 'biotype']
    """
    n_spots = rna_counts.shape[0]
    genes = gene_meta["hgnc_symbol"].astype(str).values
    chroms = gene_meta["chrom"].astype(str).values
    starts = gene_meta["start"].values
    ends = gene_meta["end"].values
    biotypes = gene_meta["biotype"].astype(str).values

    # 1. Artifact & Autosomal Protein-Coding Filter
    is_mt_ribo = np.array([
        g.startswith("MT-") or g.startswith("RPL") or g.startswith("RPS")
        for g in genes
    ])
    autosomes = {str(i) for i in range(1, 23)} | {f"chr{i}" for i in range(1, 23)}
    is_autosomal = np.array([c in autosomes for c in chroms])
    is_protein_coding = (biotypes == "protein_coding")
    spot_detection_rate = (rna_counts > 0).sum(axis=0) / float(n_spots)
    passes_min_spots = spot_detection_rate >= 0.01

    valid_mask = (~is_mt_ribo) & is_autosomal & is_protein_coding & passes_min_spots
    rna_valid = rna_counts[:, valid_mask].astype(np.float64)
    meta_valid = gene_meta.iloc[valid_mask].reset_index(drop=True)

    # 2. Log-library-size normalization on primary (+MHC) RNA
    lib_size = rna_valid.sum(axis=1, keepdims=True)
    lib_size = np.where(lib_size == 0, 1.0, lib_size)
    X_rna_plus_mhc = np.log1p(1e4 * (rna_valid / lib_size))

    # 3. Sensitivity (-MHC) Mask: exclude chr6: 25,000,000 - 34,000,000
    is_chr6 = meta_valid["chrom"].astype(str).isin(["6", "chr6"]).values
    in_mhc_window = is_chr6 & (meta_valid["end"].values >= 25_000_000) & (meta_valid["start"].values <= 34_000_000)
    minus_mhc_mask = ~in_mhc_window

    rna_minus_raw = rna_valid[:, minus_mhc_mask]
    lib_size_minus = rna_minus_raw.sum(axis=1, keepdims=True)
    lib_size_minus = np.where(lib_size_minus == 0, 1.0, lib_size_minus)
    X_rna_minus_mhc = np.log1p(1e4 * (rna_minus_raw / lib_size_minus))
    meta_minus_mhc = meta_valid.iloc[minus_mhc_mask].reset_index(drop=True)

    # 4. Surface Proteome (ADT) Centered Log-Ratio (CLR) across spots
    # X_{i,p}^{(s,ADT)} = ln(Count_{i,p} + 1) - (1/n_s) * sum_j ln(Count_{j,p} + 1)
    log_adt = np.log(adt_counts.astype(np.float64) + 1.0)
    X_adt_clr = log_adt - np.mean(log_adt, axis=0, keepdims=True)

    return {
        "X_rna_plus": X_rna_plus_mhc,
        "genes_plus": meta_valid["hgnc_symbol"].values,
        "meta_plus": meta_valid,
        "X_rna_minus": X_rna_minus_mhc,
        "genes_minus": meta_minus_mhc["hgnc_symbol"].values,
        "meta_minus": meta_minus_mhc,
        "X_adt": X_adt_clr,
        "adt_raw_pos": (adt_counts > 0).astype(np.float64),
        "adt_names": list(adt_names)
    }


# =====================================================================
# STAGE 2: INTRA-SECTION LATENT SMOOTHING & DUAL MATRIX GENERATION
# =====================================================================

def stage2_latent_smoothing_and_specificity(
    E_latent: np.ndarray,
    domain_labels: np.ndarray,
    X_feature: np.ndarray,
    raw_pos_indicator: Optional[np.ndarray] = None,
    k_micro: int = 30,
    compute_output_2a: bool = True
) -> Dict[str, object]:
    """
    Stage 2:
    - Computes cosine similarity s_{ij}^{(s)} in E^{(s)}.
    - Builds micro-domain N_i^{(s)} = {i} U top K_micro=30 neighbors with s_{ij} > 0.
    - Output 2A: Convex Cosine-Smoothed Log-Expression X_tilde (for scDRS).
    - Output 2B: Rank-Ratio Specificity Matrix GSS and Canonical Domain Vectors v_k.
    """
    n_spots, n_features = X_feature.shape
    if raw_pos_indicator is None:
        raw_pos_indicator = (X_feature > 0).astype(np.float64)

    # 3.1 Cosine Similarity Graph in E^{(s)}
    norms = np.linalg.norm(E_latent, axis=1, keepdims=True)
    norms = np.where(norms == 0, 1.0, norms)
    E_norm = E_latent / norms
    S_cos = E_norm @ E_norm.T
    np.fill_diagonal(S_cos, 1.0)

    neighbors_list = []
    W_tilde = np.zeros((n_spots, n_spots), dtype=np.float64)

    for i in range(n_spots):
        sim_i = S_cos[i].copy()
        sim_i[i] = -np.inf  # Temporarily exclude self to pick top K_micro neighbors
        pos_idx = np.where(sim_i > 0)[0]
        if len(pos_idx) > k_micro:
            top_k_sub = np.argpartition(sim_i[pos_idx], -k_micro)[-k_micro:]
            nbrs = pos_idx[top_k_sub]
        else:
            nbrs = pos_idx
        micro_domain = np.concatenate(([i], nbrs))
        neighbors_list.append(micro_domain)

        weights = S_cos[i, micro_domain]
        W_tilde[i, micro_domain] = weights / np.sum(weights)

    # 3.2 Output 2A: Convex Cosine-Smoothed Log-Expression Matrix
    X_tilde = (W_tilde @ X_feature) if compute_output_2a else None

    # 3.3 Output 2B: Across-spot ranks & Micro-domain Rank Enrichment Ratio
    R_matrix = np.zeros_like(X_feature, dtype=np.float64)
    for f in range(n_features):
        R_matrix[:, f] = stats.rankdata(X_feature[:, f], method="average")

    mean_rank_section = np.mean(R_matrix, axis=0, keepdims=True)  # (n_s + 1) / 2
    GSS = np.zeros_like(X_feature, dtype=np.float64)

    for i in range(n_spots):
        nbrs = neighbors_list[i]
        micro_mean_rank = np.mean(R_matrix[nbrs, :], axis=0)
        r_tilde_i = micro_mean_rank / mean_rank_section[0]
        det_frac_i = np.mean(raw_pos_indicator[nbrs, :], axis=0)

        active = (r_tilde_i > 1.0) & (det_frac_i >= 0.05)
        GSS[i, active] = np.exp(r_tilde_i[active]) - 1.0

    # Canonical Domain Specificity Vectors v_k^{(s, mu)}
    unique_domains = np.unique(domain_labels)
    v_domain = {}
    for dom in unique_domains:
        idx = np.where(domain_labels == dom)[0]
        v_domain[dom] = np.mean(GSS[idx, :], axis=0)

    return {
        "X_tilde": X_tilde,
        "GSS": GSS,
        "v_domain": v_domain,
        "neighbors_list": neighbors_list,
        "W_tilde": W_tilde
    }


# =====================================================================
# STAGE 3: THREE DISEASE MAPPING ENGINES (BRANCH A: A1 & D1)
# =====================================================================

def engine1_scdrs_bottom_up(
    X_tilde_rna: np.ndarray,
    genes: np.ndarray,
    magma_z_dict: Dict[str, pd.Series],
    n_ctrl: int = 1000,
    n_bins: int = 20,
    seed: int = 42
) -> Dict[str, Tuple[np.ndarray, np.ndarray]]:
    """
    Stage 3.2 Engine 1: Bottom-Up Transcriptomic Polygenic Risk (MAGMA + scDRS on Output 2A).
    Matches top 1,000 putative disease genes (w_{g,d} = min(10, max(0, z))) against B=1,000
    Monte Carlo control gene sets binned into a 20 x 20 mean-expression and variance grid.
    """
    rng = np.random.default_rng(seed)
    n_spots, n_genes = X_tilde_rna.shape
    gene_to_idx = {g: idx for idx, g in enumerate(genes)}

    # Compute 20 x 20 across-spot mean and variance bins
    gene_means = np.mean(X_tilde_rna, axis=0)
    gene_vars = np.var(X_tilde_rna, axis=0, ddof=1)

    mean_bins = pd.qcut(gene_means, q=n_bins, labels=False, duplicates="drop")
    bin_2d = np.zeros(n_genes, dtype=int)
    for mb in np.unique(mean_bins):
        idx_mb = np.where(mean_bins == mb)[0]
        if len(idx_mb) >= n_bins:
            vb = pd.qcut(gene_vars[idx_mb], q=n_bins, labels=False, duplicates="drop")
        else:
            vb = np.zeros(len(idx_mb), dtype=int)
        bin_2d[idx_mb] = mb * n_bins + vb

    bin_to_genes = {b: np.where(bin_2d == b)[0] for b in np.unique(bin_2d)}

    results = {}
    for trait, z_series in magma_z_dict.items():
        # Filter to genes present in section and positive z-score
        common_genes = [g for g in z_series.index if g in gene_to_idx and z_series[g] > 0]
        if len(common_genes) == 0:
            results[trait] = (np.zeros(n_spots), np.ones(n_spots))
            continue

        z_vals = np.array([z_series[g] for g in common_genes])
        top_order = np.argsort(z_vals)[::-1][:1000]
        sel_idx = np.array([gene_to_idx[common_genes[k]] for k in top_order])
        weights = np.clip(z_vals[top_order], 0.0, 10.0)
        weights_norm = weights / np.sum(weights)

        # Raw disease score per spot
        raw_score = X_tilde_rna[:, sel_idx] @ weights_norm

        # Draw B=1,000 Monte Carlo control gene sets matched by 20x20 bin
        ctrl_scores = np.zeros((n_spots, n_ctrl), dtype=np.float64)
        sel_bins = bin_2d[sel_idx]
        ctrl_gene_matrix = np.zeros((len(sel_idx), n_ctrl), dtype=int)
        for k_g, b_id in enumerate(sel_bins):
            pool = bin_to_genes[b_id]
            ctrl_gene_matrix[k_g, :] = rng.choice(pool, size=n_ctrl, replace=True)

        for b in range(n_ctrl):
            ctrl_scores[:, b] = X_tilde_rna[:, ctrl_gene_matrix[:, b]] @ weights_norm

        # Variance-standardize per spot and across spots per scDRS protocol
        ctrl_mean_spot = np.mean(ctrl_scores, axis=1)
        ctrl_sd_spot = np.std(ctrl_scores, axis=1, ddof=1)
        ctrl_sd_spot = np.where(ctrl_sd_spot == 0, 1.0, ctrl_sd_spot)

        z_spot = (raw_score - ctrl_mean_spot) / ctrl_sd_spot
        ctrl_z = (ctrl_scores - ctrl_mean_spot[:, None]) / ctrl_sd_spot[:, None]

        # One-sided empirical spot p-value
        p_spot = (1.0 + np.sum(ctrl_z >= z_spot[:, None], axis=1)) / (1.0 + n_ctrl)
        results[trait] = (z_spot, p_spot)

    return results


def engine2_top_down_domain_enrichment(
    v_domain_rna: Dict[object, np.ndarray],
    genes: np.ndarray,
    magma_z_dict: Dict[str, pd.Series],
    gene_covariates: pd.DataFrame,
    sldsc_results_lookup: Optional[Dict[Tuple[object, str], Tuple[float, float]]] = None
) -> Dict[Tuple[object, str], Dict[str, float]]:
    """
    Stage 3.3 Engine 2: Top-Down Domain Heritability & Gene-Property Enrichment (S-LDSC + MAGMA).
    - Normalizes positive domain specificity weights v_k^{(s,RNA)} to [0,1] via min-max quantile scaling.
    - Runs competitive MAGMA regression adjusting for C_cov (gene length, log SNP density,
      inverse MAC, baseline expression).
    - Ingests S-LDSC slope tau_hat and P_LDSC (or runs pre-linked SNP annotation regression)
      and combines P_LDSC and P_MAGMA via unweighted ACAT.
    """
    gene_to_idx = {g: i for i, g in enumerate(genes)}
    results = {}

    for dom, v_vec in v_domain_rna.items():
        # Min-max quantile scaling of positive weights onto [0, 1]
        v_scaled = np.zeros_like(v_vec, dtype=np.float64)
        pos_mask = v_vec > 0
        if np.any(pos_mask):
            ranks = stats.rankdata(v_vec[pos_mask], method="average")
            if len(ranks) > 1:
                v_scaled[pos_mask] = (ranks - np.min(ranks)) / (np.max(ranks) - np.min(ranks))
            else:
                v_scaled[pos_mask] = 1.0

        for trait, z_series in magma_z_dict.items():
            common = [g for g in genes if g in z_series.index and g in gene_covariates.index]
            idx_sec = [gene_to_idx[g] for g in common]
            y_z = z_series.loc[common].values.astype(np.float64)
            C_cov = gene_covariates.loc[common].values.astype(np.float64)
            v_sub = v_vec[idx_sec].astype(np.float64)

            # Design matrix: [1, C_cov, v_k]
            X_des = np.column_stack([np.ones(len(common)), C_cov, v_sub])
            # GLS / OLS estimator for beta_{k,d}^{(s,MAGMA)}
            xtx_inv = np.linalg.pinv(X_des.T @ X_des)
            beta_hat_all = xtx_inv @ (X_des.T @ y_z)
            residuals = y_z - X_des @ beta_hat_all
            dof = max(1, len(common) - X_des.shape[1])
            sigma2 = np.sum(residuals ** 2) / dof
            se_beta = np.sqrt(max(1e-12, sigma2 * xtx_inv[-1, -1]))

            beta_magma = float(beta_hat_all[-1])
            t_stat = beta_magma / se_beta
            p_magma = float(1.0 - stats.t.cdf(t_stat, df=dof))

            # Retrieve S-LDSC tau_hat and one-sided p-value (if external S-LDSC output provided, else use scaled SNP-annotation proxy)
            if sldsc_results_lookup is not None and (dom, trait) in sldsc_results_lookup:
                tau_ldsc, p_ldsc = sldsc_results_lookup[(dom, trait)]
            else:
                # Fallback linear heritability slope on normalized [0,1] specificity weights
                X_ldsc = np.column_stack([np.ones(len(common)), C_cov, v_scaled[idx_sec]])
                xtx_inv_l = np.linalg.pinv(X_ldsc.T @ X_ldsc)
                b_l = xtx_inv_l @ (X_ldsc.T @ y_z)
                res_l = y_z - X_ldsc @ b_l
                s2_l = np.sum(res_l ** 2) / dof
                se_l = np.sqrt(max(1e-12, s2_l * xtx_inv_l[-1, -1]))
                tau_ldsc = float(b_l[-1])
                p_ldsc = float(1.0 - stats.norm.cdf(tau_ldsc / se_l))

            p_e2 = acat_combine(np.array([p_ldsc, p_magma]))
            delta_e2_pos = (tau_ldsc > 0.0) and (beta_magma > 0.0)

            results[(dom, trait)] = {
                "tau_ldsc": tau_ldsc,
                "p_ldsc": p_ldsc,
                "beta_magma": beta_magma,
                "p_magma": p_magma,
                "p_e2": p_e2,
                "delta_e2": 1.0 if delta_e2_pos else -1.0
            }

    return results


def engine3b_surface_proteome_gwas(
    PSS_adt: np.ndarray,
    adt_names: List[str],
    magma_z_dict: Dict[str, pd.Series],
    adt_to_genes: Dict[str, List[str]] = DEFAULT_ADT_TO_GENES,
    eps: float = 1e-8
) -> Dict[str, np.ndarray]:
    """
    Stage 3.4 Branch 3B: Human Lymph Node (A1, D1) — Pure Surface Proteome GWAS Engine.
    - Maps each antibody p to encoding gene(s) G(p) and computes w_{p,d}^{(s,GWAS)}.
    - Standardizes PSS_{i,p}^{(s,ADT)} across spots to Z_{i,p}^{(s,ADT)}.
    - Accounts for empirical protein correlation matrix Sigma_ADT^{(s)}.
    """
    n_spots, n_proteins = PSS_adt.shape

    # Standardize PSS across spots to zero mean and unit variance
    pss_mean = np.mean(PSS_adt, axis=0, keepdims=True)
    pss_std = np.std(PSS_adt, axis=0, ddof=1, keepdims=True)
    pss_std = np.where(pss_std == 0, 1.0, pss_std)
    Z_adt = (PSS_adt - pss_mean) / pss_std

    # Empirical spot correlation matrix among the P_s surface proteins
    Sigma_adt = np.corrcoef(Z_adt, rowvar=False)
    Sigma_adt = np.nan_to_num(Sigma_adt, nan=0.0)
    np.fill_diagonal(Sigma_adt, 1.0)

    z_e3_dict = {}
    for trait, z_series in magma_z_dict.items():
        z_p_raw = np.zeros(n_proteins, dtype=np.float64)
        for p_idx, p_name in enumerate(adt_names):
            mapped_genes = adt_to_genes.get(p_name, [p_name])
            present_z = [z_series[g] for g in mapped_genes if g in z_series.index]
            z_p_raw[p_idx] = max(present_z) if len(present_z) > 0 else 0.0

        pos_z = np.maximum(0.0, z_p_raw)
        w_p = pos_z / (np.sum(pos_z) + eps)

        denom_var = float(w_p.T @ Sigma_adt @ w_p)
        if denom_var <= 1e-12:
            z_e3_dict[trait] = np.zeros(n_spots, dtype=np.float64)
        else:
            z_e3_dict[trait] = (Z_adt @ w_p) / np.sqrt(denom_var)

    return z_e3_dict


# =====================================================================
# STAGE 4A.3: POST-HOC RESTRICTED SPATIAL REGRESSION (NNGP-RSR AUDIT)
# =====================================================================

def post_hoc_restricted_spatial_regression(
    z_spot_e1: np.ndarray,
    domain_labels: np.ndarray,
    target_domain: object,
    spatial_coords: np.ndarray,
    m_nn: int = 15
) -> Tuple[float, float]:
    """
    Stage 4A.3: Hodges-Reich Restricted Spatial Regression (RSR) with Matérn (nu=3/2) NNGP.
    Projects the spatial random effect onto the orthogonal complement P_perp = I - X_k (X_k^T X_k)^{-1} X_k^T
    so beta_{k,d}^{RSR} is unconfounded while SE(beta_{k,d}^{RSR}) accounts for spatial autocorrelation.
    """
    n_spots = len(z_spot_e1)
    ind_k = (domain_labels == target_domain).astype(np.float64)
    X_k = np.column_stack([np.ones(n_spots), ind_k])

    # Orthogonal projection matrix P_perp = I - X_k (X_k^T X_k)^{-1} X_k^T
    xtx_inv = np.linalg.pinv(X_k.T @ X_k)
    beta_rsr_vec = xtx_inv @ (X_k.T @ z_spot_e1)
    beta_rsr = float(beta_rsr_vec[1])

    # Residuals u + eps in the orthogonal complement space
    resid = z_spot_e1 - X_k @ beta_rsr_vec

    # Estimate Matérn (nu=3/2) length scale l_s from median nearest-neighbor distance
    dist_mat = cdist(spatial_coords, spatial_coords)
    np.fill_diagonal(dist_mat, np.inf)
    nn_dists = np.partition(dist_mat, min(m_nn, n_spots - 1), axis=1)[:, :min(m_nn, n_spots - 1)]
    ell_s = float(np.median(nn_dists) * 2.0)
    ell_s = max(ell_s, 1e-6)
    np.fill_diagonal(dist_mat, 0.0)

    # Sparse NNGP Matérn (nu=3/2) covariance approximation over m_nn spatial neighbors
    sigma_u2 = float(np.var(resid, ddof=2)) * 0.8
    sigma_eps2 = float(np.var(resid, ddof=2)) * 0.2

    sqrt3_h_ell = (np.sqrt(3.0) * dist_mat) / ell_s
    Sigma_matern = sigma_u2 * (1.0 + sqrt3_h_ell) * np.exp(-sqrt3_h_ell)

    # Retain top m_nn spatial neighbors per spot for NNGP sparsity
    knn_mask = np.zeros_like(Sigma_matern, dtype=bool)
    for i in range(n_spots):
        nearest = np.argsort(dist_mat[i])[: (m_nn + 1)]
        knn_mask[i, nearest] = True
    knn_mask = knn_mask | knn_mask.T
    Sigma_nngp = np.where(knn_mask, Sigma_matern, 0.0) + np.eye(n_spots) * sigma_eps2

    # Inflated variance of beta_RSR accounting for spatial covariance within domains
    # Sandwich variance estimator under restricted spatial covariance
    meat = X_k.T @ Sigma_nngp @ X_k
    cov_beta = xtx_inv @ meat @ xtx_inv
    se_beta_rsr = float(np.sqrt(max(1e-12, cov_beta[1, 1])))

    z_stat = beta_rsr / se_beta_rsr
    p_rsr = float(1.0 - stats.norm.cdf(z_stat))
    return beta_rsr, p_rsr


# =====================================================================
# STAGE 5.1: SPATIAL TOROIDAL SHIFT & MORAN SPECTRAL RANDOMIZATION
# =====================================================================

def generate_toroidal_shifted_domains(
    spatial_coords: np.ndarray,
    domain_labels: np.ndarray,
    n_perm: int = 1000,
    seed: int = 42
) -> np.ndarray:
    """
    Stage 5.1.1: 2D Toroidal Shift + Rotation on obsm['spatial'].
    Returns shifted domain labels of shape (n_perm, n_spots).
    """
    rng = np.random.default_rng(seed)
    n_spots = spatial_coords.shape[0]
    center = np.mean(spatial_coords, axis=0, keepdims=True)
    coords_centered = spatial_coords - center
    mins = np.min(coords_centered, axis=0)
    maxs = np.max(coords_centered, axis=0)
    span = np.where((maxs - mins) == 0, 1.0, maxs - mins)

    shifted_labels = np.empty((n_perm, n_spots), dtype=domain_labels.dtype)
    for b in range(n_perm):
        theta = rng.uniform(0.0, 2.0 * np.pi)
        rot_mat = np.array([
            [np.cos(theta), -np.sin(theta)],
            [np.sin(theta),  np.cos(theta)]
        ])
        delta = rng.uniform(mins, maxs)
        # Rotate and wrap on periodic bounding torus
        shifted_coords = ((coords_centered @ rot_mat.T + delta - mins) % span) + mins
        # Reassign each original spot to the nearest shifted coordinate's label
        d_mat = cdist(coords_centered, shifted_coords)
        nearest_idx = np.argmin(d_mat, axis=1)
        shifted_labels[b, :] = domain_labels[nearest_idx]

    return shifted_labels


# =====================================================================
# STAGE 4A, 4B & 5.2: AGGREGATION, TRIANGULATION & RECALIBRATION
# =====================================================================

def run_section_triangulation_and_calibration(
    domain_labels: np.ndarray,
    spatial_coords: np.ndarray,
    e1_results: Dict[str, Tuple[np.ndarray, np.ndarray]],
    e2_results: Dict[Tuple[object, str], Dict[str, float]],
    e3_z_dict: Dict[str, np.ndarray],
    trait_panels: Dict[str, List[str]] = TRAIT_PANELS_LYMPH_NODE,
    n_perm_initial: int = 1000
) -> Dict[str, object]:
    """
    Executes Stage 4A, Stage 4B, Stage 5.1, and Stage 5.2 (Strict Negative-Control
    Falsification Gate & Mandatory Empirical Permutation Recalibration if triggered).
    """
    unique_domains = np.unique(domain_labels)
    all_traits = (
        trait_panels["D_disc"]
        + trait_panels["D_neg"]
        + trait_panels["D_spec"]
    )

    def _evaluate_with_permutations(n_perm: int, force_empirical_calibration: bool):
        shifted_doms = generate_toroidal_shifted_domains(
            spatial_coords, domain_labels, n_perm=n_perm, seed=101
        )

        # Compute Engine 3B empirical spot p-values against spatial spin null
        e3_p_spot_dict = {}
        for trait, z_vec in e3_z_dict.items():
            # Surrogate null distribution across spatially shifted configurations
            null_vals = z_vec[np.argsort(shifted_doms, axis=1)]
            p_spot_e3 = (1.0 + np.sum(null_vals >= z_vec[None, :], axis=0)) / (1.0 + n_perm)
            e3_p_spot_dict[trait] = p_spot_e3

        raw_records = {}
        p_spin_dict = {}
        emp_p_engine = {}

        for dom in unique_domains:
            in_mask = (domain_labels == dom)
            out_mask = ~in_mask

            for trait in all_traits:
                z_e1, p_e1_spot = e1_results[trait]
                z_e3 = e3_z_dict[trait]
                p_e3_spot = e3_p_spot_dict[trait]

                # Observed ACAT domain p-values and test statistics
                p_e1_obs = acat_combine(p_e1_spot[in_mask])
                t_e1_obs = acat_statistic_only(p_e1_spot[in_mask])

                p_e2_obs = e2_results[(dom, trait)]["p_e2"]
                t_e2_obs = acat_statistic_only(np.array([
                    e2_results[(dom, trait)]["p_ldsc"],
                    e2_results[(dom, trait)]["p_magma"]
                ]))

                p_e3_obs = acat_combine(p_e3_spot[in_mask])
                t_e3_obs = acat_statistic_only(p_e3_spot[in_mask])

                # Directional effect statistics (Stage 4A.2)
                delta_e1 = float(np.mean(z_e1[in_mask]) - np.mean(z_e1[out_mask]))
                delta_e2 = float(e2_results[(dom, trait)]["delta_e2"])
                delta_e3 = float(np.mean(z_e3[in_mask]) - np.mean(z_e3[out_mask]))

                # Null domain statistics across B_perm toroidal shifts
                t_e1_null = np.zeros(n_perm, dtype=np.float64)
                t_e3_null = np.zeros(n_perm, dtype=np.float64)
                t_omni_null = np.zeros(n_perm, dtype=np.float64)

                for b in range(n_perm):
                    b_mask = (shifted_doms[b] == dom)
                    if not np.any(b_mask):
                        b_mask = in_mask
                    p1_b = acat_combine(p_e1_spot[b_mask])
                    p3_b = acat_combine(p_e3_spot[b_mask])
                    t_e1_null[b] = acat_statistic_only(p_e1_spot[b_mask])
                    t_e3_null[b] = acat_statistic_only(p_e3_spot[b_mask])
                    t_omni_null[b] = acat_statistic_only(np.array([p1_b, p_e2_obs, p3_b]))

                t_omni_obs = acat_statistic_only(np.array([p_e1_obs, p_e2_obs, p_e3_obs]))
                p_spin = float((1.0 + np.sum(t_omni_null >= t_omni_obs)) / (1.0 + n_perm))
                p_spin_dict[(dom, trait)] = p_spin

                if force_empirical_calibration:
                    # Stage 5.2.1: Replace analytical engine p-values with empirical exceedance probabilities
                    p1_use = float((1.0 + np.sum(t_e1_null >= t_e1_obs)) / (1.0 + n_perm))
                    p2_use = p_e2_obs
                    p3_use = float((1.0 + np.sum(t_e3_null >= t_e3_obs)) / (1.0 + n_perm))
                    t_omni_emp_obs = acat_statistic_only(np.array([p1_use, p2_use, p3_use]))
                    p_omni_use = float((1.0 + np.sum(t_omni_null >= t_omni_emp_obs)) / (1.0 + n_perm))
                else:
                    p1_use, p2_use, p3_use = p_e1_obs, p_e2_obs, p_e3_obs
                    p_omni_use = acat_combine(np.array([p1_use, p2_use, p3_use]))

                raw_records[(dom, trait)] = {
                    "domain": dom,
                    "trait": trait,
                    "p_E1": p1_use,
                    "p_E2": p2_use,
                    "p_E3": p3_use,
                    "p_omni": p_omni_use,
                    "delta_E1": delta_e1,
                    "delta_E2": delta_e2,
                    "delta_E3": delta_e3,
                    "p_spin": p_spin
                }

        # Stage 4B.1: Within-Engine & Omnibus BH-FDR across K_s x |D_disc|
        # (Negative controls D_neg are evaluated against the exact same thresholds)
        for panel_name in ["D_disc", "D_neg", "D_spec"]:
            panel_traits = trait_panels[panel_name]
            pairs = [(dom, tr) for dom in unique_domains for tr in panel_traits]
            for col_p, col_q in [("p_E1", "q_E1"), ("p_E2", "q_E2"), ("p_E3", "q_E3"), ("p_omni", "q_omni")]:
                p_arr = np.array([raw_records[pair][col_p] for pair in pairs])
                _, q_arr, _, _ = multipletests(p_arr, alpha=0.05, method="fdr_bh")
                for pair, q_val in zip(pairs, q_arr):
                    raw_records[pair][col_q] = float(q_val)

        # Stage 4B.2: Locked Cross-Axis Triangulation Decision Rules + Stage 4A.3 RSR Audit
        for pair, rec in raw_records.items():
            dom, trait = pair
            d1_pos = rec["delta_E1"] > 0
            d2_pos = rec["delta_E2"] > 0
            d3_pos = rec["delta_E3"] > 0

            path_a = (
                (rec["q_E1"] < 0.05 and d1_pos)
                and (rec["q_E2"] < 0.05 and d2_pos)
                and (rec["p_E3"] < 0.05 and d3_pos)
            )
            path_b = (
                ((rec["q_E1"] < 0.05 and d1_pos) or (rec["q_E2"] < 0.05 and d2_pos))
                and (rec["q_E3"] < 0.05 and d3_pos)
            )

            prelim_confirmed = (
                (rec["q_omni"] < 0.05)
                and (path_a or path_b)
                and (rec["p_spin"] < 0.05)
            )

            if prelim_confirmed:
                beta_rsr, p_rsr = post_hoc_restricted_spatial_regression(
                    e1_results[trait][0], domain_labels, dom, spatial_coords
                )
                rec["beta_RSR"] = beta_rsr
                rec["p_RSR"] = p_rsr
                if p_rsr < 0.05 and beta_rsr > 0:
                    rec["tier"] = "Confirmed"
                else:
                    rec["tier"] = "Tentative"
            else:
                rec["beta_RSR"] = np.nan
                rec["p_RSR"] = np.nan
                any_engine_fdr = (
                    (rec["q_E1"] < 0.05 and d1_pos)
                    or (rec["q_E2"] < 0.05 and d2_pos)
                    or (rec["q_E3"] < 0.05 and d3_pos)
                )
                if (rec["q_omni"] < 0.10) and (rec["p_spin"] < 0.05) and any_engine_fdr:
                    rec["tier"] = "Tentative"
                else:
                    rec["tier"] = "Negative"

        return pd.DataFrame(list(raw_records.values()))

    # Initial Pass (Analytical Engine p-values + B_perm = 1,000 spin check)
    df_res = _evaluate_with_permutations(n_perm=n_perm_initial, force_empirical_calibration=False)

    # Stage 5.2: Strict Negative-Control Falsification Gate Check
    neg_hits = df_res[
        (df_res["trait"].isin(trait_panels["D_neg"])) & (df_res["tier"] == "Confirmed")
    ]
    recalibrated = False
    if len(neg_hits) > 0:
        # Trigger Mandatory Empirical Permutation Recalibration with B_perm = 10,000
        recalibrated = True
        df_res = _evaluate_with_permutations(n_perm=10000, force_empirical_calibration=True)

    # Compute M_eff diagnostic on Primary Discovery Panel
    df_disc = df_res[df_res["trait"].isin(trait_panels["D_disc"])]
    p_grid = df_disc[["p_E1", "p_E2", "p_E3"]].values
    m_eff = compute_meff_cheverud_nyholt(p_grid)

    return {
        "results_df": df_res,
        "M_eff": m_eff,
        "empirical_recalibration_triggered": recalibrated
    }


def apply_mhc_sensitivity_tags(
    df_plus_mhc: pd.DataFrame,
    df_minus_mhc: pd.DataFrame
) -> pd.DataFrame:
    """
    Stage 4B.3: Mandatory MHC Sensitivity Classification (A1 & D1).
    Tags Confirmed/Tentative hits as [MHC-Robust Polygenic] or [MHC-Driven].
    """
    df_merged = df_plus_mhc.copy()
    minus_lookup = df_minus_mhc.set_index(["domain", "trait"])

    tags = []
    for _, row in df_merged.iterrows():
        key = (row["domain"], row["trait"])
        if row["tier"] in ("Confirmed", "Tentative") and key in minus_lookup.index:
            m_row = minus_lookup.loc[key]
            d1_pos = m_row["delta_E1"] > 0
            d2_pos = m_row["delta_E2"] > 0
            d3_pos = m_row["delta_E3"] > 0
            path_a_m = (
                (m_row["q_E1"] < 0.05 and d1_pos)
                and (m_row["q_E2"] < 0.05 and d2_pos)
                and (m_row["p_E3"] < 0.05 and d3_pos)
            )
            path_b_m = (
                ((m_row["q_E1"] < 0.05 and d1_pos) or (m_row["q_E2"] < 0.05 and d2_pos))
                and (m_row["q_E3"] < 0.05 and d3_pos)
            )
            if (m_row["q_omni"] < 0.05) and (path_a_m or path_b_m):
                tags.append("[MHC-Robust Polygenic]")
            else:
                tags.append("[MHC-Driven]")
        else:
            tags.append("N/A (Null)")

    df_merged["mhc_class"] = tags
    return df_merged


# =====================================================================
# STAGE 4C: CANONICAL CROSS-DATASET MATCHING & DONOR REPLICATION (A1 vs D1)
# =====================================================================

def stage4c_canonical_donor_matching_and_replication(
    v_rna_A1: Dict[object, np.ndarray],
    genes_A1: np.ndarray,
    v_adt_A1: Dict[object, np.ndarray],
    adt_A1: List[str],
    df_res_A1: pd.DataFrame,
    v_rna_D1: Dict[object, np.ndarray],
    genes_D1: np.ndarray,
    v_adt_D1: Dict[object, np.ndarray],
    adt_D1: List[str],
    df_res_D1: pd.DataFrame,
    tau_match: float = 0.40
) -> Dict[str, object]:
    """
    Stage 4C.1 & 4C.2:
    - Intersects top 2,000 shared highly variable HGNC genes from v_k^{(s,RNA)}
      concatenated with P shared surface proteins from v_k^{(s,ADT)} (standardized within modality).
    - Computes K_A1 x K_D1 cross-section Spearman correlation matrix M^{(A1, D1)}.
    - Links homologous canonical domains iff (k, k') in Hungarian_Match INTERSECT RBH AND rho >= 0.40.
    - Declares 'Replicated & Confirmed' iff Confirmed independently in both A1 and D1.
    """
    doms_A1 = list(v_rna_A1.keys())
    doms_D1 = list(v_rna_D1.keys())

    # 1. Shared top 2,000 highly variable genes across domain specificity vectors
    shared_genes = np.intersect1d(genes_A1, genes_D1)
    idx_g_A1 = [np.where(genes_A1 == g)[0][0] for g in shared_genes]
    idx_g_D1 = [np.where(genes_D1 == g)[0][0] for g in shared_genes]

    mat_rna_A1 = np.vstack([v_rna_A1[d][idx_g_A1] for d in doms_A1])
    mat_rna_D1 = np.vstack([v_rna_D1[d][idx_g_D1] for d in doms_D1])

    combined_var = np.var(mat_rna_A1, axis=0) + np.var(mat_rna_D1, axis=0)
    top_2000_idx = np.argsort(combined_var)[::-1][:2000]

    rna_sub_A1 = stats.zscore(mat_rna_A1[:, top_2000_idx], axis=1, nan_policy="omit")
    rna_sub_D1 = stats.zscore(mat_rna_D1[:, top_2000_idx], axis=1, nan_policy="omit")

    # 2. Shared surface proteins standardized within modality
    shared_adt = [p for p in adt_A1 if p in adt_D1]
    idx_p_A1 = [adt_A1.index(p) for p in shared_adt]
    idx_p_D1 = [adt_D1.index(p) for p in shared_adt]

    mat_adt_A1 = np.vstack([v_adt_A1[d][idx_p_A1] for d in doms_A1])
    mat_adt_D1 = np.vstack([v_adt_D1[d][idx_p_D1] for d in doms_D1])

    adt_sub_A1 = stats.zscore(mat_adt_A1, axis=1, nan_policy="omit")
    adt_sub_D1 = stats.zscore(mat_adt_D1, axis=1, nan_policy="omit")

    feat_A1 = np.nan_to_num(np.hstack([rna_sub_A1, adt_sub_A1]), nan=0.0)
    feat_D1 = np.nan_to_num(np.hstack([rna_sub_D1, adt_sub_D1]), nan=0.0)

    # 3. Cross-section Spearman correlation matrix M^{(A1, D1)}
    M_corr = np.zeros((len(doms_A1), len(doms_D1)), dtype=np.float64)
    for i in range(len(doms_A1)):
        for j in range(len(doms_D1)):
            rho, _ = stats.spearmanr(feat_A1[i], feat_D1[j])
            M_corr[i, j] = float(np.nan_to_num(rho, nan=0.0))

    # 4. Hungarian Algorithm + Reciprocal Best Hit (RBH) at rho >= 0.40
    row_ind, col_ind = linear_sum_assignment(-M_corr)
    hungarian_set = set(zip(row_ind, col_ind))

    best_for_A1 = np.argmax(M_corr, axis=1)
    best_for_D1 = np.argmax(M_corr, axis=0)
    rbh_set = {
        (i, j)
        for i in range(len(doms_A1))
        for j in range(len(doms_D1))
        if best_for_A1[i] == j and best_for_D1[j] == i
    }

    canonical_matches = []
    for (i, j) in sorted(hungarian_set.intersection(rbh_set)):
        if M_corr[i, j] >= tau_match:
            canonical_matches.append({
                "canonical_id": f"C_{len(canonical_matches) + 1}",
                "domain_A1": doms_A1[i],
                "domain_D1": doms_D1[j],
                "spearman_rho": M_corr[i, j]
            })

    df_matches = pd.DataFrame(canonical_matches)

    # 5. Independent Donor Replication Table
    lookup_A1 = df_res_A1.set_index(["domain", "trait"])
    lookup_D1 = df_res_D1.set_index(["domain", "trait"])
    all_traits = df_res_A1["trait"].unique()

    replication_rows = []
    for _, m_row in df_matches.iterrows():
        c_id = m_row["canonical_id"]
        d_a1 = m_row["domain_A1"]
        d_d1 = m_row["domain_D1"]
        rho_val = m_row["spearman_rho"]

        for trait in all_traits:
            r_a1 = lookup_A1.loc[(d_a1, trait)]
            r_d1 = lookup_D1.loc[(d_d1, trait)]

            if r_a1["tier"] == "Confirmed" and r_d1["tier"] == "Confirmed":
                final_tier = "Replicated & Confirmed"
            elif r_a1["tier"] == "Confirmed" or r_d1["tier"] == "Confirmed":
                final_tier = "Unreplicated / Donor-Specific Candidate"
            elif r_a1["tier"] == "Tentative" or r_d1["tier"] == "Tentative":
                final_tier = "Tentative"
            else:
                final_tier = "Negative"

            replication_rows.append({
                "canonical_id": c_id,
                "domain_A1": d_a1,
                "domain_D1": d_d1,
                "homologous_rho": rho_val,
                "trait": trait,
                "tier_A1": r_a1["tier"],
                "q_omni_A1": r_a1["q_omni"],
                "mhc_class_A1": r_a1.get("mhc_class", "N/A"),
                "tier_D1": r_d1["tier"],
                "q_omni_D1": r_d1["q_omni"],
                "mhc_class_D1": r_d1.get("mhc_class", "N/A"),
                "final_replication_tier": final_tier
            })

    return {
        "M_corr": pd.DataFrame(M_corr, index=doms_A1, columns=doms_D1),
        "canonical_matches": df_matches,
        "replication_table": pd.DataFrame(replication_rows)
    }


# =====================================================================
# STAGE 5.3.2: POST-HOC GROUND-TRUTH HISTOLOGICAL ALIGNMENT
# =====================================================================

def stage5_ground_truth_alignment(
    domain_labels: np.ndarray,
    manual_anno: np.ndarray
) -> Dict[str, object]:
    """
    Stage 5.3.2: Evaluates contingency table between unsupervised domains C^(s)
    and held-out ground-truth annotations (annotation.csv column manual-anno)
    using ARI, NMI, and per-domain Jaccard overlap.
    """
    ari = float(adjusted_rand_score(manual_anno, domain_labels))
    nmi = float(normalized_mutual_info_score(manual_anno, domain_labels))

    doms = np.unique(domain_labels)
    annos = np.unique(manual_anno)
    jaccard_mat = np.zeros((len(doms), len(annos)), dtype=np.float64)

    for i, d in enumerate(doms):
        d_mask = (domain_labels == d)
        for j, a in enumerate(annos):
            a_mask = (manual_anno == a)
            inter = np.sum(d_mask & a_mask)
            union = np.sum(d_mask | a_mask)
            jaccard_mat[i, j] = float(inter / union) if union > 0 else 0.0

    return {
        "ARI": ari,
        "NMI": nmi,
        "jaccard_table": pd.DataFrame(jaccard_mat, index=doms, columns=annos)
    }



In [ ]:
# ---------- DRIVER (verbatim from hello.md, block 2 of 2; ONLY edit: n_perm_initial=REAL_N_PERM) ----------
import scanpy as sc

def run_single_lymph_node_section(
    section_id: str,
    rna_h5ad_path: str,
    adt_h5ad_path: str,
    embedding_E: np.ndarray,
    domain_C: np.ndarray,
    annotation_csv_path: str,
    magma_z_dict: Dict[str, pd.Series],
    gene_covariates: pd.DataFrame,
    sldsc_lookup_plus: Optional[Dict] = None,
    sldsc_lookup_minus: Optional[Dict] = None
) -> Dict[str, object]:
    """Runs Stages 1, 2, 3, 4A, 4B, and 5 for a single lymph node section (A1 or D1)."""
    adata_rna = sc.read_h5ad(rna_h5ad_path)
    adata_adt = sc.read_h5ad(adt_h5ad_path)
    spatial_coords = np.asarray(adata_rna.obsm["spatial"], dtype=np.float64)

    rna_counts = adata_rna.X.toarray() if hasattr(adata_rna.X, "toarray") else np.asarray(adata_rna.X)
    adt_counts = adata_adt.X.toarray() if hasattr(adata_adt.X, "toarray") else np.asarray(adata_adt.X)

    gene_meta = pd.DataFrame({
        "hgnc_symbol": adata_rna.var_names.values,
        "chrom": adata_rna.var["chrom"].values,
        "start": adata_rna.var["start"].values,
        "end": adata_rna.var["end"].values,
        "biotype": adata_rna.var["biotype"].values
    })

    # Stage 1: Harmonization (+MHC and -MHC tracks + CLR ADT)
    s1 = stage1_harmonize_lymph_node(
        rna_counts=rna_counts,
        gene_meta=gene_meta,
        adt_counts=adt_counts,
        adt_names=list(adata_adt.var_names)
    )

    # Stage 2: Smoothing & Specificity (+MHC RNA, -MHC RNA, and ADT)
    s2_rna_plus = stage2_latent_smoothing_and_specificity(
        E_latent=embedding_E,
        domain_labels=domain_C,
        X_feature=s1["X_rna_plus"],
        compute_output_2a=True
    )
    s2_rna_minus = stage2_latent_smoothing_and_specificity(
        E_latent=embedding_E,
        domain_labels=domain_C,
        X_feature=s1["X_rna_minus"],
        compute_output_2a=True
    )
    s2_adt = stage2_latent_smoothing_and_specificity(
        E_latent=embedding_E,
        domain_labels=domain_C,
        X_feature=s1["X_adt"],
        raw_pos_indicator=s1["adt_raw_pos"],
        compute_output_2a=False
    )

    # Stage 3: Run 3 Engines on Primary (+MHC) Track
    e1_plus = engine1_scdrs_bottom_up(s2_rna_plus["X_tilde"], s1["genes_plus"], magma_z_dict)
    e2_plus = engine2_top_down_domain_enrichment(
        s2_rna_plus["v_domain"], s1["genes_plus"], magma_z_dict, gene_covariates, sldsc_lookup_plus
    )
    e3_z = engine3b_surface_proteome_gwas(s2_adt["GSS"], s1["adt_names"], magma_z_dict)

    # Stage 3: Run Engines on Sensitivity (-MHC) Track
    e1_minus = engine1_scdrs_bottom_up(s2_rna_minus["X_tilde"], s1["genes_minus"], magma_z_dict)
    e2_minus = engine2_top_down_domain_enrichment(
        s2_rna_minus["v_domain"], s1["genes_minus"], magma_z_dict, gene_covariates, sldsc_lookup_minus
    )

    # Stage 4A, 4B & 5.2: Triangulation & Calibration on Primary (+MHC) and Sensitivity (-MHC)
    tri_plus = run_section_triangulation_and_calibration(
        domain_C, spatial_coords, e1_plus, e2_plus, e3_z, n_perm_initial=REAL_N_PERM
    )
    tri_minus = run_section_triangulation_and_calibration(
        domain_C, spatial_coords, e1_minus, e2_minus, e3_z, n_perm_initial=REAL_N_PERM
    )

    # Stage 4B.3: Tag [MHC-Robust Polygenic] vs [MHC-Driven]
    df_final_section = apply_mhc_sensitivity_tags(tri_plus["results_df"], tri_minus["results_df"])

    # Stage 5.3.2: Post-hoc Ground-Truth Histological Alignment (locked after disease calls)
    anno_df = pd.read_csv(annotation_csv_path)
    gt_eval = stage5_ground_truth_alignment(domain_C, anno_df["manual-anno"].values)

    return {
        "section_id": section_id,
        "results_df": df_final_section,
        "M_eff": tri_plus["M_eff"],
        "recalibrated": tri_plus["empirical_recalibration_triggered"],
        "v_domain_rna": s2_rna_plus["v_domain"],
        "genes_plus": s1["genes_plus"],
        "v_domain_adt": s2_adt["v_domain"],
        "adt_names": s1["adt_names"],
        "ground_truth_eval": gt_eval
    }



In [ ]:
# ---------- PANEL OVERRIDE (pilot): FL and DLBCL as separate traits ----------
# hello.md's panel names a combined "Follicular_Lymphoma_DLBCL" trait, but no
# combined GWAS exists - MAGMA produced separate DLBCL and FL z-scores.
# The triangulation reads TRAIT_PANELS_LYMPH_NODE at call time, so mutating
# D_disc here is enough; D_neg and D_spec are untouched.
TRAIT_PANELS_LYMPH_NODE["D_disc"] = ["SLE", "RA", "IBD", "MS", "T1D", "Celiac",
                                     "DLBCL", "FL", "CLL", "Lymphocyte_Count"]
_all = (TRAIT_PANELS_LYMPH_NODE["D_disc"] + TRAIT_PANELS_LYMPH_NODE["D_neg"]
        + TRAIT_PANELS_LYMPH_NODE["D_spec"])
assert set(_all) == set(magma_z_dict.keys()), (set(_all) ^ set(magma_z_dict.keys()))
print("panel traits (16):", _all)


In [ ]:
# ---------- Run one section + save + checkpoint ----------
import time, json
RES = {}
def run_and_save(sec):
    t0 = time.time()
    res = run_single_lymph_node_section(
        section_id=sec,
        rna_h5ad_path=SEC_PREP[sec]["rna"],
        adt_h5ad_path=SEC_PREP[sec]["adt"],
        embedding_E=EMB[sec]["E"],
        domain_C=EMB[sec]["C"],
        annotation_csv_path=SEC_PREP[sec]["anno"],
        magma_z_dict=magma_z_dict,
        gene_covariates=gene_covariates,
        sldsc_lookup_plus=None,     # as-is: Engine 2 internal OLS-proxy fallback
        sldsc_lookup_minus=None,
    )
    RES[sec] = res
    df = res["results_df"]
    mins = (time.time() - t0) / 60
    print(f"{sec}: run finished in {mins:.1f} min | results_df {df.shape}")
    print("tier counts:\n", df["tier"].value_counts())
    print("M_eff:", res["M_eff"], "| recalibrated:", res["recalibrated"])
    print("GT post-hoc: ARI =", res["ground_truth_eval"]["ARI"], "| NMI =", res["ground_truth_eval"]["NMI"])
    neg = df[df["trait"].isin(TRAIT_PANELS_LYMPH_NODE["D_neg"])]
    print("NEGATIVE CONTROLS (read these first):")
    print(neg[["domain", "trait", "tier", "q_omni"]].to_string(index=False))
    df.to_csv(os.path.join(OUTDIR, f"{sec}_results_df.csv"), index=False)
    res["ground_truth_eval"]["jaccard_table"].to_csv(os.path.join(OUTDIR, f"{sec}_gt_jaccard.csv"))
    summary = {
        "run": "hello-pipeline REAL-DATA PILOT - real MAGMA z + covariates + GENCODE v44 annotation; engines as-is (E2 OLS proxy, no Pp RSR, E3 code null); FL/DLBCL separate traits (16 total)",
        "section": sec, "n_perm_initial": REAL_N_PERM,
        "runtime_min": round(mins, 1),
        "E_shape": list(EMB[sec]["E"].shape), "n_domains": int(len(np.unique(EMB[sec]["C"]))),
        "M_eff": float(res["M_eff"]), "recalibrated": bool(res["recalibrated"]),
        "tier_counts": {k: int(v) for k, v in df["tier"].value_counts().items()},
        "gt_ARI": float(res["ground_truth_eval"]["ARI"]), "gt_NMI": float(res["ground_truth_eval"]["NMI"]),
    }
    with open(os.path.join(OUTDIR, f"{sec}_summary.json"), "w") as f:
        json.dump(summary, f, indent=2)
    push_outputs(f"pilot-real: {sec} results (real MAGMA, as-is engines)")
    return res


In [ ]:
res_a1 = run_and_save("A1")

In [ ]:
res_d1 = run_and_save("D1")

In [ ]:
# ---------- Stage 4C: canonical matching + replication (A1 x D1) ----------
s4c = stage4c_canonical_donor_matching_and_replication(
    RES["A1"]["v_domain_rna"], RES["A1"]["genes_plus"], RES["A1"]["v_domain_adt"], RES["A1"]["adt_names"], RES["A1"]["results_df"],
    RES["D1"]["v_domain_rna"], RES["D1"]["genes_plus"], RES["D1"]["v_domain_adt"], RES["D1"]["adt_names"], RES["D1"]["results_df"],
    tau_match=0.40,
)
s4c["M_corr"].to_csv(os.path.join(OUTDIR, "stage4c_M_corr.csv"))
s4c["canonical_matches"].to_csv(os.path.join(OUTDIR, "stage4c_canonical_matches.csv"), index=False)
s4c["replication_table"].to_csv(os.path.join(OUTDIR, "stage4c_replication_table.csv"), index=False)
print("canonical matches:")
print(s4c["canonical_matches"].to_string(index=False))
print("final replication tiers:\n", s4c["replication_table"]["final_replication_tier"].value_counts())
push_outputs("pilot-real: Stage 4C matching + replication (A1 x D1)")
print("PILOT RUN COMPLETE - outputs in", OUTDIR)
